# Évaluation WESAD sur Google Colab — autonome, sans rien télécharger chez soi

WESAD pèse **2,5 Go**. Ce notebook le télécharge **sur les serveurs de Google**, calcule la VFC (RMSSD) par fenêtre de 60 s pour les 15 participants, évalue la règle `HRV-01` d'Astro Assistant, puis ne renvoie que **deux petits fichiers** :

- `wesad_hrv_windows.csv` (quelques centaines de Ko) → à mettre dans `backend/data/physionet/` ; le seed l'importe en base ;
- `calibration_wesad.json` → sensibilité et taux de fausses alertes de la règle VFC.

Il ne dépend pas du dépôt GitHub (le code de calcul est inclus, identique à `backend/app/engine/calibration.py`).

**Mode d'emploi** : https://colab.research.google.com → *Fichier → Importer le notebook* → onglet *Importer* → choisir ce fichier, puis *Exécution → Tout exécuter*. Durée : 10 à 15 minutes.

**Conditions d'utilisation de WESAD** : usage scientifique non commercial, en citant Schmidt et al., *Introducing WESAD, a Multimodal Dataset for Wearable Stress and Affect Detection*, ICMI 2018.

## 1. Télécharger et décompresser WESAD (côté Google)

In [ ]:
# Lien officiel publié par l'université de Siegen (partage Sciebo).
!wget -c -q --show-progress -O /content/WESAD.zip "https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download"
!unzip -q -o /content/WESAD.zip -d /content/wesad_raw
import pathlib

files = sorted(pathlib.Path("/content/wesad_raw").rglob("S*.pkl"))
print(len(files), "participants :", [f.stem for f in files])

## 2. Calcul de la VFC (RMSSD) par fenêtre de 60 s — même code que le backend

In [ ]:
import csv
import pickle
from statistics import median

import numpy as np
from scipy.signal import butter, filtfilt, find_peaks

ECG_HZ, WINDOW_S = 700, 60
CONDITIONS = {1: "baseline", 2: "stress", 3: "amusement"}


def rr_intervals(ecg, fs=ECG_HZ):
    b, a = butter(3, [5 / (fs / 2), 15 / (fs / 2)], btype="band")
    energy = filtfilt(b, a, ecg) ** 2
    peaks, _ = find_peaks(energy, distance=int(0.33 * fs), height=np.percentile(energy, 90))
    rr = np.diff(peaks) / fs * 1000
    return rr[(rr > 333) & (rr < 1500)]


def rmssd(rr):
    if len(rr) < 20:
        return None
    diff = np.diff(rr)
    diff = diff[np.abs(diff) < 200]
    return float(np.sqrt(np.mean(diff**2))) if len(diff) >= 10 else None


rows = []
for path in files:
    with path.open("rb") as handle:
        data = pickle.load(handle, encoding="latin1")
    ecg = np.asarray(data["signal"]["chest"]["ECG"]).ravel()
    labels = np.asarray(data["label"]).ravel()
    step = ECG_HZ * WINDOW_S
    for start in range(0, len(ecg) - step, step):
        window = labels[start : start + step]
        label = int(window[0])
        if label not in CONDITIONS or not (window == label).all():
            continue
        value = rmssd(rr_intervals(ecg[start : start + step]))
        if value is not None:
            rows.append([path.stem, CONDITIONS[label], start / ECG_HZ, round(value, 3)])
    print(path.stem, "ok")

with open("/content/wesad_hrv_windows.csv", "w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["subject", "condition", "t_offset_s", "rmssd_ms"])
    writer.writerows(rows)
print(len(rows), "fenêtres écrites")

## 3. Évaluer la règle HRV-01 (VFC < seuil × baseline individuelle)

In [ ]:
import json

subjects = {}
for subject, condition, _, value in rows:
    subjects.setdefault(subject, []).append((condition, value))

results = []
for ratio in (0.6, 0.7, 0.8, 0.9):
    tp = fn = fp = tn = 0
    for windows in subjects.values():
        base = [v for c, v in windows if c == "baseline"]
        if len(base) < 5:
            continue
        reference = median(base)
        for condition, value in windows:
            flagged = value < ratio * reference
            if condition == "stress":
                tp, fn = tp + flagged, fn + (not flagged)
            else:
                fp, tn = fp + flagged, tn + (not flagged)
    results.append(
        {
            "ratio": ratio,
            "sensitivity": round(tp / (tp + fn), 3),
            "false_alarm_rate": round(fp / (fp + tn), 3),
            "stress_windows": tp + fn,
            "non_stress_windows": fp + tn,
        }
    )
    print(
        f"seuil {ratio:.0%} → sensibilité {tp / (tp + fn):.0%}, fausses alertes {fp / (fp + tn):.0%}"
    )

stress_ratio = [
    median([v for c, v in w if c == "stress"]) / median([v for c, v in w if c == "baseline"])
    for w in subjects.values()
    if any(c == "stress" for c, _ in w) and sum(c == "baseline" for c, _ in w) >= 5
]
report = {
    "dataset": "WESAD (Univ. Siegen), ECG thoracique 700 Hz, fenêtres de 60 s",
    "subjects": len(subjects),
    "median_rmssd_stress_over_baseline": round(median(stress_ratio), 3),
    "hrv_rule": results,
}
json.dump(
    {"wesad": report},
    open("/content/calibration_wesad.json", "w", encoding="utf-8"),
    ensure_ascii=False,
    indent=2,
)
print(json.dumps(report, ensure_ascii=False, indent=2))

## 4. Récupérer les deux petits fichiers

In [ ]:
from google.colab import files as colab_files

colab_files.download("/content/wesad_hrv_windows.csv")
colab_files.download("/content/calibration_wesad.json")

Ensuite, sur ton ordinateur : copie `wesad_hrv_windows.csv` dans `backend/data/physionet/` et `calibration_wesad.json` dans `backend/docs/`, puis préviens l'équipe : `python -m app.seed --reset` les met en base.